# Voronoi geometry → 2D conformal mesh

Build a 2D Voronoi polycrystal with `vtpolyxtal2d` (Shapely seeds), then mesh
it with `confMesh2dGMSH.from_geometric_pxtal` / `femesh_gmsh`.

This is the Voronoi **geometry source**, not Technique B (`confMesh2d_technique_b.ipynb`,
which unions Voronoi cells that share an LFI label).

Requires `gmsh` (`pip install upxo[mesh]`). `confMesh2d` (pygmsh) is deprecated.
Canonical mesh-only demo: `confMesh2d_gmsh.ipynb`.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from upxo.geoEntities.mulpoint2d import MPoint2d
from upxo.pxtal.polyxtal import vtpolyxtal2d
from upxo.meshing.conformal_mesher2d import confMesh2dGMSH
from upxo.meshing.writer_ABQ import summarize_inp


In [ ]:
rng = np.random.default_rng(1)
xy = rng.uniform(1.5, 8.5, size=(12, 2))
mpo = MPoint2d.from_coords(xy)
gtess = vtpolyxtal2d(
    gsgen_method='vt', vt_base_tool='shapely',
    mulpoint_object=mpo, xbound=(0, 10), ybound=(0, 10), vis_vtgs=False)
print('grains', gtess.L0.xtals_n)

`from_geometric_pxtal` is the drop-in for the old pygmsh constructor. Bounds must match the tessellation domain.

In [ ]:
m = confMesh2dGMSH.from_geometric_pxtal(
    pxtal=gtess.L0.pxtal, xbound=(0, 10), ybound=(0, 10))
m.femesh_gmsh(mesh_size_gb=0.45, mesh_size_bulk=1.1,
              mesh_algo=6, recombine_to_quads=False)
m.form_elsets_gmsh(); m.build_boundary_nsets(); m.build_gb_nset()
print(m.validation_report)
fig, ax = m.plot_by_grain(figsize=(6, 6), show_gb=True, show_nsets=True,
                          title='Voronoi vtpolyxtal2d')
fig

In [ ]:
out = Path.cwd() / 'confMesh2d_voronoi_out'
inp = m.export_abaqus_inp(out / 'rve_cps3.inp', plane='stress')
inp, summarize_inp(inp)